这是**层次分析法 (AHP, Analytic Hierarchy Process)** 的详细解析。它是数学建模中最基础、最常用的评价与决策算法，特别适合**“主观性强、没有现成数据、需要拍脑袋决定权重”**的场景。

---

### 一、 算法原理
层次分析法的核心思想是将一个复杂的决策问题分解为三个层次：
1.  **目标层**：你的最终目的是什么？（例如：选最佳旅游地）。
2.  **准则层**：你考虑哪些因素？（例如：景色、花费、居住、饮食）。
3.  **方案层**：你有哪几个选择？（例如：苏杭、北戴河、桂林）。

**原理**：人很难直接给所有因素同时打分，但人很擅长**“两两比较”**。AHP通过让你对比“A和B谁更重要”，利用数学方法推算出所有因素的权重，并进行一致性检验（防止你逻辑混乱，比如你说A比B强，B比C强，结果又说C比A强）。

---

### 二、 推导与步骤

#### 1. 构建判断矩阵 (Judgment Matrix)
使用 **1-9 标度法** 对指标进行两两比较：
*   1：两者同样重要
*   3：前者比后者 稍微重要
*   5：前者比后者 明显重要
*   7：前者比后者 强烈重要
*   9：前者比后者 极端重要
*   2, 4, 6, 8：上述判断的中间值
*   倒数：若 A 比 B 重要是 3，则 B 比 A 重要就是 1/3。

假设有三个指标 $A, B, C$，判断矩阵 $X$ 长这样：
$$
X = \begin{bmatrix}
1 & 3 & 5 \\
1/3 & 1 & 2 \\
1/5 & 1/2 & 1
\end{bmatrix}
$$

#### 2. 计算权重 (算术平均法)
虽然有特征值法（数学上更严谨），但**算术平均法**（Sum-Product Method）在建模比赛中更常用且好解释：
1.  **列向量归一化**：每一列的数除以该列之和。
2.  **求行和**：将归一化后的矩阵按行求和。
3.  **归一化**：将行和除以指标个数 $n$，得到权重向量 $w$。

#### 3. 一致性检验 (非常重要！)
必须检验你的判断逻辑是否自相矛盾。
1.  计算最大特征值 $\lambda_{max} = \sum_{i=1}^{n} \frac{(Aw)_i}{n \times w_i}$
2.  计算一致性指标 $CI = \frac{\lambda_{max} - n}{n - 1}$
3.  查询平均随机一致性指标 $RI$（查表，见代码）。
4.  计算一致性比例 $CR = \frac{CI}{RI}$。
    *   **若 $CR < 0.1$**：通过检验，权重有效。
    *   **若 $CR \ge 0.1$**：逻辑混乱，需要修改判断矩阵。

---

### 三、 适用场景
1.  **评价类问题**：选优秀员工、选最佳方案。
2.  **赋权问题**：当没有具体数据（如熵权法无法使用），只能凭专家经验判断各指标重要性时，用AHP求权重。
3.  **决策问题**：买房、买车、填报志愿（综合考虑价格、性能、距离等）。

---

### 四、 Python 代码框架

这份代码封装了一个 `AHP` 类。你只需要把你的**判断矩阵**填进去，它就会自动计算权重并告诉你是否通过检验。

```python
import numpy as np

class AHP:
    def __init__(self, criteria_matrix):
        """
        criteria_matrix: 用户输入的判断矩阵 (numpy array)
        """
        self.matrix = np.array(criteria_matrix)
        self.n = self.matrix.shape[0] # 指标个数
        self.RI_dict = {1: 0, 2: 0, 3: 0.58, 4: 0.90, 5: 1.12,
                        6: 1.24, 7: 1.32, 8: 1.41, 9: 1.45} # RI查表标准值

    def calculate_weights(self):
        """
        使用算术平均法计算权重
        """
        # 1. 列向量归一化
        column_sum = np.sum(self.matrix, axis=0)
        standardized_matrix = self.matrix / column_sum

        # 2. 求行和
        row_sum = np.sum(standardized_matrix, axis=1)

        # 3. 归一化得到权重
        self.weights = row_sum / self.n
        return self.weights

    def consistency_check(self):
        """
        进行一致性检验
        """
        # 计算最大特征值 lambda_max
        # lambda_max = sum((Aw)_i / (n * w_i))
        Aw = np.dot(self.matrix, self.weights)
        lambda_max = np.sum(Aw / (self.n * self.weights)) / self.n

        # 计算 CI
        CI = (lambda_max - self.n) / (self.n - 1)

        # 查找 RI
        if self.n in self.RI_dict:
            RI = self.RI_dict[self.n]
        else:
            print(f"警告: 矩阵维度 n={self.n} 超出常见RI表范围，请手动查表补充。")
            RI = 1.45 # 默认给个大值防止报错

        # 计算 CR
        # 只有当 n > 2 时才需要计算CR，n=1或2时CI为0，必定通过
        if self.n <= 2:
            CR = 0.0
        else:
            CR = CI / RI

        print("-" * 30)
        print(f"最大特征值 lambda_max: {lambda_max:.4f}")
        print(f"一致性指标 CI: {CI:.4f}")
        print(f"一致性比例 CR: {CR:.4f}")

        if CR < 0.1:
            print(">> 结果: 通过一致性检验！权重有效。")
            return True
        else:
            print(">> 结果: 未通过一致性检验 (CR >= 0.1)。请调整判断矩阵！")
            return False

# ================= 使用示例 =================

if __name__ == "__main__":
    # 场景模拟：我们要评价“景色、花费、居住、饮食”4个指标的权重
    # 用户(专家)给出的判断矩阵：
    #      景  花  居  食
    # 景 [[1,  2,  5,  7],  -> 景色比花费稍重要(2)，比居住明显重要(5)...
    # 花  [1/2,1,  3,  5],
    # 居  [1/5,1/3,1,  2],
    # 食  [1/7,1/5,1/2,1]]

    # 注意：矩阵必须是正互反矩阵 (对角线为1，a_ij = 1 / a_ji)
    input_matrix = np.array([
        [1,   2,   5,   7],
        [1/2, 1,   3,   5],
        [1/5, 1/3, 1,   2],
        [1/7, 1/5, 1/2, 1]
    ])

    # 1. 实例化模型
    ahp_model = AHP(input_matrix)

    # 2. 计算权重
    weights = ahp_model.calculate_weights()

    # 3. 进行检验
    is_valid = ahp_model.consistency_check()

    # 4. 输出最终结果
    if is_valid:
        print("-" * 30)
        print("最终计算的指标权重:")
        indicators = ['景色', '花费', '居住', '饮食'] # 修改成你的指标名
        for i, w in enumerate(weights):
            print(f"{indicators[i]}: {w:.4f} ({w*100:.2f}%)")
```

### 如何“修修补补”使用：
1.  **替换 `input_matrix`**：这是你唯一需要大量修改的地方。根据你的题目，自己构造这个矩阵。
    *   *技巧*：如果检验不通过（CR > 0.1），不需要重新思考人生，只需要微调矩阵里的数字。比如把 `5` 改成 `3`，或者把 `1/7` 改成 `1/5`，通常就能通过了。
2.  **替换 `indicators` 列表**：把名字换成你论文里的指标名称，方便直接截图结果。
3.  **多层级怎么办？**：如果你的题目有“准则层”和“方案层”，你需要**跑多次代码**。
    *   第1次：输入准则层的矩阵，算出准则的权重（比如：景色0.5，花费0.3...）。
    *   第2次：针对“景色”这一项，输入各旅游地的矩阵，算出在景色维度的得分。
    *   ...
    *   最后：手动用Excel或者加几行代码进行加权求和（综合评分 = 准则权重 × 方案得分）。